# DATA255 MobileNetV3-Small — Colab

Enable free T4 GPU, upload nathan_shared.zip and nih_rgb224.zip in the Files sidebar, then run the cells in order. The PNG archive is a lossless cache of the team's exact RGB + 224×224 bilinear resize. Its pixel hashes and 16 original images are checked before training. Random flip and ImageNet normalization remain in the original train.py.

Nathan's shared five CSVs match the bundled ones as dataframes, and his patched train.py is byte-identical to the supplied local script. SETTINGS='patched' preserves the later patched defaults (seed 2026, weight decay 0.0001); SETTINGS='original' selects the earlier message's seed 42 and weight decay 0.01. All runs use ImageNet pretraining, 10 epochs, batch 32, lr 1e-4, and the same validation/test data.

Results and epoch checkpoints live on this runtime until downloaded. The final cell downloads the results archive.


In [ ]:
from pathlib import Path
import base64, gzip, json, subprocess, sys
WORK = Path('/content/data255')
WORK.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
for name, content in json.loads(gzip.decompress(base64.b64decode(PAYLOAD))).items():
    destination = WORK / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(content))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
print('Prepared:', WORK)


In [ ]:
from pathlib import Path
import torch, torchvision, timm, zipfile
IMAGE_ROOT = Path('/content/nih_images')
# Prefer Nathan's downloaded shared input; bundled CSVs were checked to match.
shared_zip = Path('/content/nathan_shared.zip')
DATA_DIR = WORK / 'data'
TRAIN_SCRIPT = WORK / 'train.py'
if shared_zip.exists():
    shared_dir = WORK / 'nathan_shared'
    shared_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(shared_zip) as z:
        assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
        z.extractall(shared_dir)
    DATA_DIR = shared_dir
    TRAIN_SCRIPT = shared_dir / 'train.py'
SETTINGS = 'patched'
FRACTIONS = ['100', '50', '25']
OUTPUT_ROOT = Path('/content/data255_results')
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', torch.__version__, torchvision.__version__, timm.__version__)
print('CSV directory:', DATA_DIR)


In [ ]:
# Verify all lossless PNGs and compare original images with torchvision Resize.
import zipfile, hashlib, json
import numpy as np
from PIL import Image
from torchvision import transforms
archive = Path('/content/nih_rgb224.zip')
assert archive.exists(), 'Upload nih_rgb224.zip using the Files sidebar first.'
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall('/content')
manifest = json.loads(Path('/content/image_cache_manifest.json').read_text())
for name, expected in manifest['pixel_sha256'].items():
    with Image.open(IMAGE_ROOT / name) as im:
        assert hashlib.sha256(im.convert('RGB').tobytes()).hexdigest() == expected, name
checked = 0
for original in Path('/content/original_checks').glob('*.png'):
    with Image.open(original) as im:
        expected = np.array(transforms.Resize((224, 224))(im.convert('RGB')))
    with Image.open(IMAGE_ROOT / original.name) as im:
        actual = np.array(im.convert('RGB'))
    assert np.array_equal(actual, expected), original.name
    checked += 1
assert checked == 16
print('Verified', manifest['images'], 'cached images and', checked, 'original/torchvision comparisons.', flush=True)
command = [sys.executable, '-u', str(WORK / 'run_mobilenet.py'),
           '--train-script', str(TRAIN_SCRIPT), '--data-dir', str(DATA_DIR),
           '--image-root', str(IMAGE_ROOT), '--output-root', str(OUTPUT_ROOT),
           '--settings', SETTINGS, '--fractions', *FRACTIONS]
subprocess.run(command + ['--preflight-only'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first. Kaggle may require phone verification.'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run(command, check=True)


In [ ]:
import shutil
from google.colab import files
print((OUTPUT_ROOT / 'all_run_summaries.json').read_text())
archive = shutil.make_archive('/content/mobilenet_results', 'zip', OUTPUT_ROOT)
files.download(archive)
